In [1]:
!pip install -q langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb pypdf


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
import chromadb
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   # ← langchain.text_splitter 에서 변경
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# ── 엔드포인트 ─────────────────────────────────────────────
LMSTUDIO_BASE_URL = "http://host.docker.internal:12345/v1"  # /v1 까지 붙여야 함
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 → 더미 값

EMBED_MODEL = "embedding-8b:sl"
LLM_MODEL   = "qwen3.6-35b:mm"

CHROMA_HOST = "chromadb"
CHROMA_PORT = 8000
COLLECTION  = "kb2024"

PDF_PATH = "2024 KB 부동산 보고서.pdf"  # 2024 KB 부동산 보고서
print("PDF_PATH =", PDF_PATH, "| 존재:", os.path.exists(PDF_PATH))


PDF_PATH = 2024 KB 부동산 보고서.pdf | 존재: True


In [4]:
embedding_function = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,   # ★ 토큰ID 대신 원문 문자열 전송
)

# 연결 확인: 차원 수가 찍히면 OK
_v = embedding_function.embed_query("연결 테스트")
print("임베딩 차원:", len(_v))

임베딩 차원: 4096


In [5]:
chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
print("heartbeat:", chroma_client.heartbeat())  # ns 값이 나오면 연결 OK

heartbeat: 1781786406741621959


In [8]:
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embedding_function,
)

count = vectorstore._collection.count()
if count == 0:
    print("컬렉션이 비어 있음 → 인덱싱 시작")
    documents = PyPDFLoader(PDF_PATH).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)
    print("분할된 청크 수:", len(chunks))
    vectorstore.add_documents(chunks)   # LM Studio 임베딩으로 벡터화 후 원격 적재
    print("적재 완료. 문서 수:", vectorstore._collection.count())
else:
    print("이미 적재됨. 문서 수:", count, "→ 인덱싱 건너뜀")

컬렉션이 비어 있음 → 인덱싱 시작
분할된 청크 수: 135
적재 완료. 문서 수: 135


In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

컨텍스트: {context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("placeholder", "{chat_history}"),
    ("human", "{question}"),
])

model = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key="lmstudio",   # 임의 문자열
    temperature=0.7,
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

base_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | model
    | StrOutputParser()
)

store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
# 빠른 검증 (서비스 가동 전 노트북에서 직접 확인)
resp = chain.invoke(
    {"question": "수도권 주택 매매 전망을 알려줘"},
    {"configurable": {"session_id": "nb-test"}},
)
print(resp)



제공해주신 2024 KB 부동산 보고서(수도권 주택시장 점검) 내용을 바탕으로, 수도권 주택 매매시장의 전망을 거시 환경, 시장 구조 변화, 지역별 변수로 나누어 정리해 드립니다.

### 🔍 핵심 전망 요약
2024년 수도권 주택 매매시장은 **거래 침체가 지속되는 가운데 가격 변동성은 완화되나, 지역 간 양극화는 심화**될 것으로 예상됩니다. 전반적인 금리 환경과 거시경제 지표가 지역별 인프라·산업 호재를 상쇄하는 지배적 변수로 작용할 전망입니다.

---

### 📊 주요 영향 요인 및 전망
| 구분 | 현황 및 전망 |
|:---|:---|
| **거시·금리 환경** | 고금리와 DSR 규제 지속으로 매수자의 구매 여력은 크게 회복되지 못함. 특례보금자리론 중단으로 실수요 중심의 매수세가 관망세로 전환되며 매매가격은 보합 또는 소폭 조정 국면 진입 예상. |
| **정책·규제 영향** | 재건축 규제 완화, 세제 개선 등 정부 정책으로 매도자의 기대심리는 상승 중이나, 구매력 회복이 더디어 공급-수요 간 간극이 지속될 수 있음. |
| **시장 양극화 심화** | 거래량은 월 5만 호 내외로 침체되나, 선호 지역(강남권 등)으로 수요가 집중되면서 해당 지역은 가격 안정성 유지 및 매도자 호가 조정 최소화. 비선호 지역은 하락 압력 또는 정체 국면 지속 예상. |

---

### 📍 대표 지역 동향 (보고서 기준)
- **동탄(기흥/화성)**: GTX-A 노선 개통, 반도체 산업단지(삼성디스플레이, ASML 등) 조성으로 교통·수요 호재가 존재하나, 고금리와 관망세가 주류를 이루며 매매가격은 보합세 유지 중. 지역적 호재보다 **전체 수도권 시장 심리와 금리 동향이 더 큰 변수**로 작용할 전망.
- **강남구(한강이남)**: 재건축 신속통합기획 추진 및 학군 수요 집중으로 가격 하락폭이 상대적으로 작음. 매수 수요가 프리미엄 지역으로 쏠리며 거래는 위축되나 호가는 견조하게 유지 중. 2024년에도 규제 완화 효과로 **선호도 높은 지역의 안정적 흐

In [11]:
%%writefile app.py
import os
import chromadb
import streamlit as st
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://host.docker.internal:12345/v1")
EMBED_MODEL = os.getenv("EMBED_MODEL", "embedding-8b:sl")
LLM_MODEL   = os.getenv("LLM_MODEL", "unsloth/gemma-4-e2b-it")
CHROMA_HOST = os.getenv("CHROMA_HOST", "chromadb")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
COLLECTION  = os.getenv("COLLECTION", "kb2024")
PDF_PATH    = os.getenv("PDF_PATH", "./_kb2024.pdf")


@st.cache_resource
def get_embeddings():
    return OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=LMSTUDIO_BASE_URL,
        api_key="lm-studio",
        check_embedding_ctx_length=False,   # ★ 토큰ID 대신 원문 문자열 전송
    )


@st.cache_resource
def initialize_vectorstore():
    client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    vs = Chroma(client=client, collection_name=COLLECTION, embedding_function=get_embeddings())
    if vs._collection.count() == 0:
        documents = PyPDFLoader(PDF_PATH).load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        vs.add_documents(splitter.split_documents(documents))
    return vs


@st.cache_resource
def initialize_chain():
    vectorstore = initialize_vectorstore()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

컨텍스트: {context}
"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", template),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ])
    model = ChatOpenAI(
        model=LLM_MODEL,
        base_url=LMSTUDIO_BASE_URL,
        api_key="lmstudio",
        temperature=0.7,
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    base_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["question"]))
        )
        | prompt
        | model
        | StrOutputParser()
    )

    store = {}

    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    return RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="question",
        history_messages_key="chat_history",
    )


def main():
    st.set_page_config(page_title="KB 부동산 보고서 챗봇", page_icon="\U0001F3E0")
    st.title("\U0001F3E0 KB 부동산 보고서 AI 어드바이저")
    st.caption("2024 KB 부동산 보고서 기반 질의응답 시스템")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if prompt := st.chat_input("부동산 관련 질문을 입력하세요"):
        with st.chat_message("user"):
            st.markdown(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})

        chain = initialize_chain()
        with st.chat_message("assistant"):
            with st.spinner("답변 생성 중..."):
                response = chain.invoke(
                    {"question": prompt},
                    {"configurable": {"session_id": "streamlit_session"}},
                )
            st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})


if __name__ == "__main__":
    main()

Writing app.py


In [14]:
!pip install -q streamlit


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [21]:
import subprocess, sys

proc = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
])
print("streamlit PID:", proc.pid, "→ 컨테이너 내부 0.0.0.0:8501")
# 중지하려면: proc.terminate()

streamlit PID: 2749 → 컨테이너 내부 0.0.0.0:8501




2026-06-18 12:52:29.635 Uvicorn server started on 0.0.0.0:8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.18.0.3:8501
  External URL: http://1.212.170.26:8501



In [22]:
import urllib.request, socket
print("컨테이너 IP:", socket.gethostbyname(socket.gethostname()))   # ★ NPM에 넣을 주소
print("상태코드:", urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=5).read())

컨테이너 IP: 172.18.0.3
상태코드: b'ok'
  Stopping...


In [23]:
import subprocess, time, socket
subprocess.run(["pkill", "-f", "streamlit run"])
time.sleep(2)
s = socket.socket(); r = s.connect_ex(("127.0.0.1", 8501)); s.close()
print("8501:", "아직 점유중" if r == 0 else "비었음(정리됨)")

8501: 비었음(정리됨)
